<style>
@font-face {
  font-family: 'Roboto';
  font-style: normal;
  font-weight: 600;
  src: local('Roboto Semi-Bold'), local('Roboto-SemiBold'), url('assets/Roboto-SemiBold.ttf') format('truetype');
}
</style>

<div align="center">
  <img src="assets/Jupyter_AIKit_logo.svg" width="600">
</div>

<h2 style="color:#FFFFFF; text-align:center; font-family:'Roboto', sans-serif; font-size:24px; font-weight:600; letter-spacing:0.08em;">DECISION MODELS</h2>

<h3 style="color:#44D62C; text-align:left;">Objectives of This Guide</h3>


This guide shows how to run a **decision model** locally with Razer AIKit and call it through the typed `/v1/systemone` API.

A decision model does not write text. You give it some content (the *state*) and one or more typed *questions*, and it returns a calibrated probability for every possible answer. That makes it a good fit for the quick, repeatable decisions software makes all the time: routing a support ticket, rating urgency, or checking whether a message mentions something.

By following this guide, you will:

- Learn the three question types: **choice**, **score** and **noul** (yes/no)
- See which **Decider** models are available and which one fits your GPU
- Start **Decider 2B** with `rzr-aikit`
- Make three sample API calls for each question type
- Use the `confidence` value to decide when to ask a person

<h3 style="color:#44D62C; text-align:left;">🧠 1. What Makes Decision Models Different</h3>


| | Chat model | Decision model |
|---|---|---|
| **Output** | Free text you have to parse | A probability for every option |
| **Speed** | Generates token by token | One forward pass, no generated tokens |
| **Repeatability** | Wording changes between runs | The same input gives the same numbers |
| **Confidence** | Not available | Built in, from 0 to 1 |
| **Where it runs** | Your GPU | Your GPU |

Each question is scored on its own, so adding, removing or reordering other questions in the same request never changes an answer.

<h3 style="color:#44D62C; text-align:left;">📚 2. Available Decider Models</h3>


AIKit supports three Decider models. All are Apache-2.0 licensed and built on the Qwen3.5 base models, and AIKit recognises them automatically, so the API is switched on when you run them.

| Model | Download size | GPU | Pick it for |
|---|---|---|---|
| `Mapika/decider-0.8b` | 1.5 GB | 8 GB and up | Simple routing and yes/no checks on short text |
| `Mapika/decider-2b` | 3.8 GB | 8 GB and up | A good default for classification and judgments |
| `Mapika/decider-4b` | 8.4 GB | 16 GB and up | Harder judgments and longer inputs |

The sizes are download sizes; leave extra GPU memory for the model's working cache. This guide uses **Decider 2B**. To use another size, change the model name in sections 3 and 4.

##### How to read an answer

- **Probabilities** for the options always add up to 1.
- **`confidence`** goes from 0 (every option equally likely) to 1 (all the weight on one option).
- The probabilities are calibrated across many decisions. They are a good guide, but they are not a guarantee for any single case.

<h3 style="color:#44D62C; text-align:left;">📥 3. Check Compatibility & Download the Model</h3>


Check that the model fits your hardware, then pull it from Hugging Face into your local cache.

In [ ]:
!rzr-aikit model info Mapika/decider-2b

In [ ]:
!rzr-aikit model download Mapika/decider-2b

<h3 style="color:#44D62C; text-align:left;">🚀 4. Run the Model</h3>


Start the model server. `rzr-aikit` picks settings that suit your GPU, and because this is a Decider model it also enables the `/v1/systemone` API.

The first start takes a few minutes while the model loads.

In [ ]:
!nohup rzr-aikit model run Mapika/decider-2b
# Output is available in nohup.out

Wait until the server reports it is ready:

In [ ]:
import time
import requests

BASE_URL = "http://localhost:8001"

for _ in range(90):
    try:
        if requests.get(f"{BASE_URL}/health", timeout=2).status_code == 200:
            print(f"Ready: POST {BASE_URL}/v1/systemone")
            break
    except requests.ConnectionError:
        pass
    time.sleep(5)
else:
    print("The server is not ready yet. Check nohup.out for details.")

<h3 style="color:#44D62C; text-align:left;">🔧 5. A Small Helper</h3>


Every call below sends a JSON request and prints the answers. This helper keeps the examples short.

In [ ]:
import requests

BASE_URL = "http://localhost:8001"


def ask(state, questions):
    """Send one SystemOne request and return its answers."""
    response = requests.post(
        f"{BASE_URL}/v1/systemone",
        json={"state": state, "questions": questions},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["answers"]


def show_choice(answer):
    print(f"Choice: {answer['choice']}   (confidence {answer['confidence']:.2f})")
    for option, p in sorted(answer["probabilities"].items(), key=lambda kv: -kv[1]):
        print(f"  {option:<22}{p:6.1%}")


def show_score(answer):
    print(f"Score: {answer['score']:.2f}   (confidence {answer['confidence']:.2f})")
    for level, p in answer["probabilities"].items():
        print(f"  {answer['legend'][level]:<40}{p:6.1%}")


def show_noul(answer):
    print(f"Probability of yes: {answer['noul']:.1%}")

<h3 style="color:#44D62C; text-align:left;">🎯 6. Choice: Pick One Option</h3>


Use a **choice** question to route or categorise. Give 2 to 255 options as a map of `name: description`. Write a clear description for each one, because the model reads them.

##### Sample 1: Route a support ticket

In [ ]:
TEAMS = {
    "billing": "Charges, refunds and invoices",
    "technical_support": "Product faults and setup",
    "sales": "Questions about buying",
}

answers = ask(
    "My card was charged twice for the same purchase.",
    {"team": {"type": "choice", "instructions": "Which team should handle this request?", "criteria": TEAMS}},
)
show_choice(answers["team"])

##### Sample 2: Find the intent of a message

In [ ]:
INTENTS = {
    "cancel": "The customer wants to end their subscription",
    "upgrade": "The customer wants a bigger or better plan",
    "question": "The customer is asking how something works",
    "complaint": "The customer is unhappy about something",
}

answers = ask(
    "I have been paying for the basic plan for a year and I would like the one with more storage.",
    {"intent": {"type": "choice", "instructions": "What does the customer want?", "criteria": INTENTS}},
)
show_choice(answers["intent"])

##### Sample 3: Judge the tone of a message

In [ ]:
TONES = {
    "angry": "Frustrated, upset or demanding",
    "neutral": "Matter of fact, no strong feeling",
    "friendly": "Warm, polite or grateful",
}

answers = ask(
    "Thanks so much for sorting this out so quickly, you have been really helpful!",
    {"tone": {"type": "choice", "instructions": "What is the tone of this message?", "criteria": TONES}},
)
show_choice(answers["tone"])

<h3 style="color:#44D62C; text-align:left;">📏 7. Score: Rate on an Ordered Scale</h3>


Use a **score** question for severity, priority or quality. Give 2 to 10 levels as an ordered list. The answer is the expected level, a number between the first and last level, plus the probability of each level.

##### Sample 1: How urgent is this request?

In [ ]:
URGENCY = [
    "Routine: can wait a week",
    "Soon: this week",
    "Today: needs attention today",
    "Immediately: an outage, act now",
]

answers = ask(
    "Our checkout is down for all customers right now and we are losing orders every minute.",
    {"urgency": {"type": "score", "instructions": "How urgent is this request?", "criteria": URGENCY}},
)
show_score(answers["urgency"])

##### Sample 2: How positive is this review?

In [ ]:
SENTIMENT = [
    "Very negative",
    "Negative",
    "Neutral",
    "Positive",
    "Very positive",
]

answers = ask(
    "The headset is comfortable and the sound is great, but the microphone cuts out now and then.",
    {"sentiment": {"type": "score", "instructions": "How positive is this review?", "criteria": SENTIMENT}},
)
show_score(answers["sentiment"])

##### Sample 3: How good is this reply?

In [ ]:
QUALITY = [
    "Unhelpful: does not address the question",
    "Weak: partly addresses the question",
    "Acceptable: answers the question",
    "Good: answers clearly and completely",
    "Excellent: answers clearly, completely and politely",
]

reply = (
    "Customer: How do I reset my password?\n"
    "Agent: Go to Settings, choose Security, then select Reset password. "
    "You will get an email with a link that works for one hour. Let me know if it does not arrive."
)
answers = ask(
    reply,
    {"quality": {"type": "score", "instructions": "How good is the agent's reply?", "criteria": QUALITY}},
)
show_score(answers["quality"])

<h3 style="color:#44D62C; text-align:left;">✅ 8. Noul: Yes or No</h3>


Use a **noul** question for a yes/no check. Only `instructions` is needed. The answer is the probability that the answer is yes: close to 1 means yes and close to 0 means no.

##### Sample 1: Does the message mention a receipt?

In [ ]:
answers = ask(
    "Hi, I bought the blender last week and it stopped working. I have the receipt attached if you need it.",
    {"has_receipt": {"type": "noul", "instructions": "Does the customer mention having a receipt?"}},
)
show_noul(answers["has_receipt"])

##### Sample 2: Is the customer asking for a refund?

In [ ]:
answers = ask(
    "The jacket arrived in the wrong size. Please take it back and return my money.",
    {"wants_refund": {"type": "noul", "instructions": "Is the customer asking for a refund?"}},
)
show_noul(answers["wants_refund"])

##### Sample 3: Does the message include an order number?

In [ ]:
answers = ask(
    "Hello, I am still waiting for my parcel. It was meant to arrive on Monday.",
    {"has_order_number": {"type": "noul", "instructions": "Does the message include an order number?"}},
)
show_noul(answers["has_order_number"])

<h3 style="color:#44D62C; text-align:left;">🧩 9. Several Questions in One Request</h3>


Put as many questions as you need under `questions`. They share one request, and each is answered on its own.

In [ ]:
answers = ask(
    "Hi, my invoice shows a charge I don't recognise from the 3rd. "
    "I need this fixed before Friday's payroll run. Order #48213.",
    {
        "team": {"type": "choice", "instructions": "Which team should handle this request?", "criteria": TEAMS},
        "urgency": {"type": "score", "instructions": "How urgent is this request?", "criteria": URGENCY},
        "has_order_number": {"type": "noul", "instructions": "Does the message include an order number?"},
    },
)
show_choice(answers["team"])
print()
show_score(answers["urgency"])
print()
show_noul(answers["has_order_number"])

<h3 style="color:#44D62C; text-align:left;">🚦 10. Use the Confidence</h3>


Probabilities let your code decide how much to trust an answer. A simple pattern is to act automatically when the model is confident and hand the case to a person when it is not.

In [ ]:
def route(message, threshold=0.8):
    answers = ask(
        message,
        {"team": {"type": "choice", "instructions": "Which team should handle this request?", "criteria": TEAMS}},
    )
    team = answers["team"]
    if team["confidence"] >= threshold:
        return f"-> {team['choice']} (confidence {team['confidence']:.2f})"
    return f"-> send to a person (confidence {team['confidence']:.2f})"


for message in [
    "My card was charged twice for the same purchase.",
    "I would like to talk to someone about my account.",
]:
    print(message)
    print(" ", route(message))

<h3 style="color:#44D62C; text-align:left;">🛑 11. Stop the Model</h3>


Shut the model down to free the GPU.

In [ ]:
!rzr-aikit model stop

---

<h3 style="color:#44D62C; text-align:left;">✅ Summary</h3>

You can run a decision model locally with Razer AIKit and called it through `/v1/systemone`.

##### Key highlights:

- Started `Mapika/decider-2b` with `rzr-aikit model run`, which switched on the SystemOne API automatically
- Asked **choice**, **score** and **noul** questions and read the probability of every answer
- Combined several questions in one request
- Used `confidence` to send uncertain cases to a person

##### Tips for good results:

- **Describe your options.** The model reads each option's description, so `billing: charges, refunds and invoices` works far better than a bare `billing`.
- **Keep the state relevant.** Include the text the decision depends on and leave out the rest.
- **Ask one thing per question.** Split a compound question into separate questions.
- **Choose your own thresholds.** Check the results on your own data before deciding where to draw the line.
- **Limits.** Up to 255 options per choice question, up to 10 levels per score question, and English text works best.